# project_24_metalloprotein — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — de novo metalloprotein design, the cofactor, and the coordination scheme

**Standard slot:** *define & explore.* **For Project 24 this means:** understand de novo
metalloprotein / cofactor-binding design, then **construct the cofactor-site spec** (the cofactor +
its coordinating ligands + the coordination geometry they must hold) and run a mock
cofactor-site→scaffold hello-world (D0).

Run `00_setup.ipynb` first in this session.

## Why a de novo cofactor-binder is *the* metalloprotein benchmark
A cofactor-binding metalloprotein holds a **redox / O₂ cofactor** — a **heme** (Fe-protoporphyrin
IX), a **[4Fe-4S]** cluster, or a **Zn** — in a protein pocket via **coordinating residues**, so the
cofactor can do its job: electron transfer, O₂ transport, or catalysis. Two things make this *the*
metalloprotein design problem:
- **It is a long-standing grand challenge.** Building a clean coordination pocket for a metal/cofactor
  from scratch is hard. The field went from hand-built four-helix-bundle **"maquettes"** (DeGrado) to
  **ML-designed** cofactor-binders (Baker lab).
- **A cofactor-aware sequence designer now exists.** **LigandMPNN** can "see" a bound heme/metal and
  design the protein around it (vanilla ProteinMPNN is cofactor-blind) — which is exactly the methods
  claim this project tests.

The honest history: designed heme proteins began as low-spin four-helix-bundle maquettes that *bound*
heme but were far from natural cytochromes; **incorporation and the designed redox/O₂ behaviour are
separate, hard-won steps** beyond a good geometry on paper.

## The cofactor-site spec — the coordination motif you must build
A **cofactor-site spec** is the minimal description of the bound cofactor plus the protein ligands
that coordinate it and the geometry they must hold. This project's **default** is a **bis-His heme**
electron-transfer site (a b-type-cytochrome motif):

| Role | Residue(s) | Job at the cofactor |
|------|-----------|---------------------|
| axial ligand 1 | His (imidazole Nε2) | coordinates the heme Fe from one face (Fe–Nε2 ~2.0–2.2 Å) |
| axial ligand 2 | His (imidazole Nε2) | coordinates the heme Fe from the opposite face (His–Fe–His ~180°) |
| (the cofactor) | heme b | the 4 porphyrin N's hold Fe in-plane; the **protein** supplies the axial ligands |

Other schemes ship in `scripts/cofactor_tools.COORDINATION_SCHEMES`: **His/Met** heme (c-type),
**proximal-His + open distal** heme (O₂-binding, myoglobin-like), **[4Fe-4S]-4Cys** (ferredoxin),
**Cys2His2** (structural Zn). You **construct** the geometry from a **verified** reference structure
and/or the literature (`data/inputs/cofactor_site_def.txt`) — it is a teaching template, **not**
fabricated data, and **there are no spectra in this project.** Place the ligands around the
**cofactor**, with the right oxidation/spin state in mind (it changes the geometry).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Build the cofactor-site spec (mock hello-world)
`scripts/cofactor_tools.py` exposes `build_cofactor_spec(cofactor, scheme)` → a coordination-pocket
geometry spec. The distances/angles it ships are **PLACEHOLDERS** — replace them in
`data/inputs/cofactor_site_def.txt` (and in `build_cofactor_spec`) with real, cited values built from
a verified reference structure during P1. This is the **enzyme-family template** hook (Project 18,
Kemp): where Project 18 placed a base + π-stack + H-bond donor and Project 20 placed a catalytic
Zn-His3-OH, Project 24 places a **cofactor + its coordinating ligands**.

In [ ]:
from cofactor_tools import build_cofactor_spec

spec = build_cofactor_spec("heme", scheme="bis_his_heme")
print("Cofactor    :", spec.cofactor, "| function:", spec.function)
print("Coordination:", spec.cofactor_site.coordination)
print("Provenance  :", spec.provenance)
print("\nCoordinating groups (PLACEHOLDER geometry — fill from a verified structure/literature):")
for g in spec.coordinating_groups:
    ang = f"{g.target_angle}deg" if g.target_angle is not None else "n/a"
    print(f"  {g.role:16s} {g.residue}/{g.atom:4s}  d={g.target_distance}A  angle={ang}")
print("\nCoordinating residues to FIX during sequence design:", spec.coordinating_residue_ids())

## A first mock scaffold + cofactor-aware sequence (no GPU)
`scaffold_cofactor_pocket(...)` (mock) returns placeholder backbones presenting the coordination
motif; `ligandmpnn_cofactor(...)` (mock) designs sequences with the coordinating residues **fixed**
and the cofactor passed as atom context. **Every number here is SYNTHETIC** — this only proves the
plumbing runs anywhere. Switch to the real backends (RFdiffusion2/Riff-Diff on an A100; cofactor-aware
LigandMPNN CPU-fast) in `02_generate.ipynb`.

In [ ]:
from cofactor_tools import scaffold_cofactor_pocket, ligandmpnn_cofactor, coordination_geometry

scaffolds = scaffold_cofactor_pocket(spec, n=5, tool="mock")
print(f"{len(scaffolds)} mock scaffolds; example:")
print(" ", scaffolds[0])

seqs = ligandmpnn_cofactor(scaffolds[0], spec.coordinating_residue_ids(), n=3,
                           cofactor=spec.cofactor, tool="mock")
print(f"\n{len(seqs)} mock sequences for {scaffolds[0]['design_id']} "
      f"(fixed roles: {seqs[0]['fixed_coordinating_roles']}, positions {seqs[0]['fixed_positions']})")

cg = coordination_geometry(None, spec)   # mock, SYNTHETIC
print(f"\ncoordination_geometry (mock, SYNTHETIC) = {cg} A  -> pass if < 0.5 A")
print("NOTE: these are placeholder numbers. The real campaign is in notebook 02.")

## The metrics that decide a metalloprotein design
| Metric | Cutoff (`enzyme`) | Means | Does **not** mean |
|--------|-------------------|-------|-------------------|
| scRMSD | ≤ 2.0 Å | designed-vs-predicted backbone self-consistency | cofactor binds |
| pLDDT (global) | ≥ 85 | local fold confidence | thermostability / coordination |
| pLDDT (**site**) | ≥ 90 | confidence *at the coordinating residues* | the geometry is correct |
| **coordination_geom_rmsd** (`cat_geom`) | **< 0.5 Å** | predicted coordinating atoms vs the target scheme | **incorporation or function** (the cofactor may not load) |

For this project the shared filter's `cat_geom` is the **coordination**-geometry RMSD and `plddt_cat`
is the **site** confidence. The fourth row is the point of the whole project — and the last column is
the message to never forget: **coordination geometry ≠ cofactor incorporation ≠ function.** A perfect
bis-His geometry on paper does not mean heme loads, and loading does not mean the redox/O₂ behaviour.
Only **spectroscopy** (UV-vis Soret / EPR + a cofactor titration) confirms it (notebook 05).

> Reminder: **AF2 does NOT place the metal/cofactor** — it predicts the apo backbone. You dock or
> superpose the cofactor to place the metal *before* scoring the coordination geometry.

## D0 checklist
- [ ] Half-page on de novo metalloprotein / cofactor-binding design + the honest hit-rate history.
- [ ] 1-page problem statement with **measurable** success criteria + the controls you'll need
      (apo protein; a coordinating-residue→Ala mutant).
- [ ] Cofactor-site spec started in `data/inputs/cofactor_site_def.txt` (replace the PLACEHOLDERs,
      cite the verified reference structure for every distance/angle + the oxidation/spin state).
- [ ] Reproduced mock hello-world (coordinating-group spec + a mock scaffold record).
- [ ] `LOG.md` entry (tool versions, GPU, seed).

**Next:** `02_generate.ipynb` — scaffold the pocket and run cofactor-aware LigandMPNN with the
coordinating residues fixed.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — cofactor site → scaffold → cofactor-aware LigandMPNN (coordinating residues fixed)

**Standard slot:** *design campaign.* **For Project 24 this means:** take the cofactor-site spec,
scaffold it into many backbones (RFdiffusion2 / Riff-Diff — the **A100** step), then **cofactor-aware
LigandMPNN sequence design fixing the coordinating residues** (and passing the cofactor as atom
context), and write a results CSV (D2).

Runs end-to-end on the **mock** backend with no GPU; switch to the real backends on Colab/HPC.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams
Tools change. Before a campaign, HTTP-check that the pinned upstream repos still exist, and pin the
commit/tag you actually use. **RFdiffusion2 and Riff-Diff are new and move fast — VERIFY the current
public release/repo at generation time** (do not assert a repo you are unsure of); the others below
are stable enough to head-check.

In [ ]:
import requests

# Pinned upstreams (pin the COMMIT/TAG you use in env/requirements.txt + LOG.md):
STABLE_UPSTREAMS = {
    "LigandMPNN (cofactor-aware, coordinating-residue-fixed seq design)": "https://github.com/dauparas/LigandMPNN",
    "RFdiffusion (classic motif scaffolding — free-tier demo)": "https://github.com/RosettaCommons/RFdiffusion",
    "ColabFold (AF2 — site pLDDT + apo backbone)": "https://github.com/sokrypton/ColabFold",
    "AutoDock Vina (cofactor fit)": "https://github.com/ccsb-scripps/AutoDock-Vina",
}
# VERIFY-ONLY (new/fast-moving; confirm the current release before relying on a URL):
VERIFY_UPSTREAMS = [
    "RFdiffusion2 (Dauparas 2025) — VERIFY current public release/repo at generation time",
    "Riff-Diff (Schnettler 2025, Nature) — VERIFY current public release/repo at generation time",
]

for name, url in STABLE_UPSTREAMS.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"[{r.status_code}] {name}\n      {url}")
    except Exception as e:
        print(f"[ERR] {name}: {e!r}\n      {url}")
print("\nVERIFY MANUALLY (do not assert a repo URL you are unsure of):")
for v in VERIFY_UPSTREAMS:
    print("  -", v)

## 1 · Build the cofactor site and scaffold its coordination pocket
The mock path returns placeholder backbones so the loop runs anywhere. On an A100, switch `METHOD` to
`"rfdiffusion2"` or `"riffdiff"` (verify the release) and `N_SCAFFOLDS` to 1000s.

> **A100 NOTE:** scaffolding 1000s of backbones is the compute bottleneck. Free Colab T4 can do a
> small **RFdiffusion** (classic) motif-scaffolding demo (tens of backbones); the real campaign wants
> an A100 (Colab Pro+) or HPC. A **cofactor pocket is harder than a single-sidechain motif** — e.g.
> two axial His on *opposite* helices at the right Fe distance/angle — so budget extra backbones
> because many will not hold a clean coordination geometry. The mock backend below needs no GPU.

In [ ]:
from cofactor_tools import build_cofactor_spec, scaffold_cofactor_pocket

COFACTOR = "heme"          # heme | fes | zn  (alias -> a coordination scheme)
SCHEME = "bis_his_heme"    # the project default; see cofactor_tools.COORDINATION_SCHEMES for others
spec = build_cofactor_spec(COFACTOR, scheme=SCHEME)

METHOD = "mock"            # -> "rfdiffusion2" | "riffdiff" | "rfdiffusion" on Colab/HPC (verify release)
N_SCAFFOLDS = 12           # -> 1000s for the real campaign

scaffolds = scaffold_cofactor_pocket(spec, n=N_SCAFFOLDS, tool=METHOD)
print(f"{len(scaffolds)} scaffolds via tool={METHOD!r}, scheme={SCHEME!r} (mock numbers are SYNTHETIC)")
print("example:", scaffolds[0])

## 2 · Cofactor-aware LigandMPNN — FIXING the coordinating residues
This is the **central tool** of the project and why **LigandMPNN, not vanilla ProteinMPNN**, is used:
the coordinating residues (e.g. the two axial His of a bis-His heme) must hold the cofactor, so (a)
those positions are **FIXED** and (b) the cofactor (heme / cluster / metal) is **passed as atom
context** so the model designs the rest of the protein to accommodate — and not clash with — the
bound cofactor. ProteinMPNN cannot see the cofactor. On Colab set `TOOL="ligandmpnn"` (CPU-fast),
pass `--ligand_mpnn_use_atom_context 1`, and a fixed-positions list covering **every** coordinating
residue.

In [ ]:
from cofactor_tools import ligandmpnn_cofactor

TOOL = "mock"              # -> "ligandmpnn" on Colab (CPU-fast)
SEQS_PER_BACKBONE = 4

coordinating = spec.coordinating_residue_ids()
all_designs = []
for bb in scaffolds:
    seqs = ligandmpnn_cofactor(bb, coordinating, n=SEQS_PER_BACKBONE,
                               cofactor=spec.cofactor, tool=TOOL)
    for s in seqs:
        s["scaffold_id"] = bb["design_id"]
        s["scaffold_tool"] = bb["tool"]
        s["motif_rmsd"] = bb["motif_rmsd"]
        all_designs.append(s)
print(f"{len(all_designs)} sequences total "
      f"({len(scaffolds)} backbones x {SEQS_PER_BACKBONE}); coordinating roles fixed: {coordinating}")
print("sanity: every sequence carries the fixed coordinating positions ->",
      all_designs[0]["fixed_positions"])

## 3 · Predict + score (mock coordination-geometry RMSD), write the results CSV
On Colab, predict each sequence with AF2/ESMFold, read the **site pLDDT** (at the coordinating
residues), **place the metal/cofactor by docking/superposition** (AF2 gives the apo backbone), and
compute the real `coordination_geometry` from the predicted PDB. Here the mock backend fills SYNTHETIC
values so the CSV — the input to notebook 03 — is produced anywhere.

In [ ]:
import pandas as pd
from cofactor_tools import coordination_geometry, dock_cofactor, cofactor_site_md

rows = []
for d in all_designs:
    # On Colab, `pred_pdb` is the AF2-predicted (apo) PDB path for this design; the mock backend keys
    # off the (non-existent) per-design path string so each design gets a DISTINCT SYNTHETIC value.
    pred_pdb = f"results/pred/{d['design_id']}.pdb"
    cg = coordination_geometry(pred_pdb, spec)         # mock -> SYNTHETIC (varies per design)
    dock = dock_cofactor(pred_pdb, spec.cofactor)      # mock -> SYNTHETIC (fit/orientation only)
    md_res = cofactor_site_md(pred_pdb, ns=10.0)       # mock -> SYNTHETIC (CAVEATED metal-FF proxy)
    # SYNTHETIC stand-ins for AF2 confidence so the plumbing runs (replace with real predictions):
    import hashlib
    h = int(hashlib.sha256(d["design_id"].encode()).hexdigest(), 16)
    plddt = 78 + (h % 20)            # 78-97, SYNTHETIC (global)
    plddt_site = 80 + ((h >> 7) % 18) # 80-97, SYNTHETIC (at the coordinating residues)
    scrmsd = round(0.8 + ((h >> 11) % 200) / 100.0, 2)  # 0.8-2.8, SYNTHETIC
    rows.append(dict(
        design_id=d["design_id"], scaffold_id=d["scaffold_id"],
        scaffold_tool=d["scaffold_tool"], cofactor=d["cofactor"], sequence=d["sequence"],
        plddt=plddt, plddt_site=plddt_site, scrmsd=scrmsd,
        coordination_geom_rmsd=cg, vina_score=dock["vina_score"],
        fe_between_ligands=dock["fe_between_ligands"], md_rmsd=md_res["md_rmsd"], synthetic=True))

camp = pd.DataFrame(rows)
camp.to_csv("results/campaign.csv", index=False)
print("wrote results/campaign.csv", camp.shape, "(ALL NUMBERS SYNTHETIC — mock backend)")
print(f"coordination_geom_rmsd range: "
      f"{camp['coordination_geom_rmsd'].min()}-{camp['coordination_geom_rmsd'].max()} A")
camp.head()

## D2 checklist
- [ ] Scaffolding run logged (tool, release/commit, N backbones, seed) — A100 for the real campaign.
- [ ] Cofactor-aware LigandMPNN sequences with the **coordinating residues provably fixed**
      (fixed-positions list logged) **and the cofactor passed as atom context**.
- [ ] `results/campaign.csv` with one row per design (real metrics on Colab; mock here).
- [ ] Design log (every config + seed + output path) + 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the shared enzyme filter on `campaign.csv`.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — the shared enzyme filter

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 24** you filter on the **enzyme** cutoffs, where `cat_geom` is the
**coordination**-geometry RMSD and `plddt_cat` is the **site** confidence — the coordination geometry
is the decisive metric.

Run `00`–`02` first so `results/campaign.csv` exists.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
Improvements here are pull-requested back to `shared/` for the whole cohort — do not silently fork it.
We reuse `design_type="enzyme"`: for a cofactor site, `cat_geom` maps to **coordination**-geometry and
`plddt_cat` to **site** confidence (pLDDT at the coordinating residues).

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("enzyme cutoffs:", fp.DEFAULT_CUTOFFS["enzyme"])
print("  (cat_geom -> coordination-geometry RMSD; plddt_cat -> site pLDDT, for this project)")

## Build `fp.Design` objects (enzyme) from the campaign
Map each campaign row onto an `fp.Design`, carrying the metalloprotein-specific fields: `plddt`, the
**site** confidence into **`plddt_catalytic`**, `scrmsd`, and the **coordination**-geometry RMSD into
**`catalytic_geom_rmsd`**. The self-consistency layer (`self_consistency`) checks all of these against
the enzyme cutoffs (scrmsd ≤ 2.0, plddt ≥ 85, plddt_cat ≥ 90, cat_geom ≤ 0.5).

In [ ]:
camp = pd.read_csv("results/campaign.csv")

designs = []
for _, r in camp.iterrows():
    designs.append(fp.Design(
        design_id=str(r["design_id"]),
        sequence=str(r.get("sequence", "")),
        design_type="enzyme",
        plddt=float(r["plddt"]),
        plddt_catalytic=float(r["plddt_site"]),          # SITE confidence -> plddt_cat
        scrmsd=float(r["scrmsd"]),
        catalytic_geom_rmsd=float(r["coordination_geom_rmsd"]),  # COORDINATION geometry -> cat_geom
        md_rmsd=float(r["md_rmsd"]),
        extra={"scaffold_tool": r["scaffold_tool"], "cofactor": r["cofactor"], "synthetic": True},
    ))
print(len(designs), "enzyme Design objects built (from SYNTHETIC mock metrics)")
print("mapping: plddt_site -> plddt_catalytic; coordination_geom_rmsd -> catalytic_geom_rmsd")

## Run the pipeline (`design_type="enzyme"`) and report
`run_pipeline` applies the layers in order and returns a ranked DataFrame. We use layers 1+3+4
(self-consistency incl. coordination geometry, physics, and the short-MD dynamics layer); the
orthogonal layer (L2) needs a second predictor's scRMSD, which you add on Colab.

In [ ]:
df_ranked = fp.run_pipeline(designs, design_type="enzyme", use_layers=(1, 3, 4))
df_ranked.to_csv("results/ranked.csv", index=False)
top = fp.report(df_ranked, top_n=10, save_prefix="results/proj24")
top

## Survival-at-each-layer + coordination-geometry pass rate (honest accounting)
The **coordination-geometry layer is where most metalloprotein designs die** — expect the steepest
drop there (a clean bis-His geometry on opposite helices is hard to scaffold). Report the pass rate
explicitly; this is the headline benchmark for D3. **And remember:** passing it does **not** mean the
cofactor incorporates — only spectroscopy decides.

In [ ]:
import pandas as pd
print("layers_passed distribution:")
print(df_ranked["layers_passed"].value_counts().sort_index())

n = len(df_ranked)
cut = fp.DEFAULT_CUTOFFS["enzyme"]["cat_geom"]
n_geom = int((df_ranked["catalytic_geom_rmsd"] <= cut).sum())
print(f"\ncoordination-geometry preservation: {n_geom}/{n} "
      f"({100*n_geom/max(n,1):.1f}%) hold the motif < {cut} A  [SYNTHETIC demo numbers]")
print("REMINDER: coordination geometry != cofactor incorporation != function. Spectroscopy decides.")

## D3 (part 1) checklist
- [ ] `results/ranked.csv` produced by the **shared** module with `design_type="enzyme"`.
- [ ] Survival-at-each-layer figure (`results/proj24_survival.png`).
- [ ] Coordination-geometry preservation rate reported (the headline metric).
- [ ] Mapping assumptions written down (site pLDDT → `plddt_cat`; coordination RMSD → `cat_geom`).

**Next:** `04_validate.ipynb` — coordination-geometry preservation + cofactor/scheme comparison +
docking / site-pLDDT / MD figures + redox-tuning reasoning.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — coordination-geometry preservation, scheme comparison, docking, site pLDDT & MD

**Standard slot:** *validate (in silico).* **For Project 24 this is the benchmark:** the
**coordination-geometry preservation rate**, a **cofactor/scheme comparison** (bis-His heme vs
His/Met heme vs [4Fe-4S] vs Zn) `[extension]`, the cofactor-docking / site-pLDDT / caveated-MD
figures, and a **redox-tuning** discussion (D3 pt2).

Needs `results/campaign.csv` (+ `results/ranked.csv` from notebook 03).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Coordination-geometry preservation — the headline figure
Distribution of coordination-geometry RMSD vs the 0.5 Å pass bar. The fraction left of the line is the
**preservation rate** — the metric that most distinguishes scaffolding tools and cofactor schemes.
(Numbers here are SYNTHETIC mock values; on Colab they come from real AF2 predictions with the
metal/cofactor placed by docking/superposition first.)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")
cut = 0.5

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(camp["coordination_geom_rmsd"], bins=20)
ax.axvline(cut, color="k", ls="--", lw=1, label=f"pass < {cut} A")
ax.set_xlabel("coordination-geometry RMSD vs target scheme (A)  [SYNTHETIC]")
ax.set_ylabel("designs"); ax.set_title("Coordination-geometry preservation")
ax.legend(); plt.tight_layout()
plt.savefig("results/coordination_geometry_hist.png", dpi=150); plt.show()

rate = 100 * (camp["coordination_geom_rmsd"] <= cut).mean()
print(f"overall coordination-geometry preservation rate = {rate:.1f}%  [SYNTHETIC demo]")
print("REALITY CHECK: a good geometry rate is NOT an incorporation rate, let alone a function rate.")

## 2 · Cofactor / scheme comparison `[extension]`
Compare the preservation rate (and hit rate) across **cofactor schemes**. In the real campaign you run
the SAME pipeline for bis-His heme vs His/Met heme vs [4Fe-4S]-4Cys vs Cys2His2 Zn (a harder
coordination is harder to scaffold cleanly); here a single mock scheme is present, so this cell shows
the *shape* of the comparison you will populate on Colab.

In [ ]:
by_cofactor = (camp.assign(pass_geom=camp["coordination_geom_rmsd"] <= 0.5)
                   .groupby("cofactor")
                   .agg(n=("design_id", "size"),
                        geom_pass_rate=("pass_geom", "mean"),
                        mean_plddt_site=("plddt_site", "mean"),
                        fe_between_ligands_rate=("fe_between_ligands", "mean"))
                   .reset_index())
by_cofactor["geom_pass_rate"] = (100 * by_cofactor["geom_pass_rate"]).round(1)
by_cofactor["fe_between_ligands_rate"] = (100 * by_cofactor["fe_between_ligands_rate"]).round(1)
print("Cofactor/scheme comparison (populate with real schemes on Colab):")
print(by_cofactor.to_string(index=False))
print("\n[SYNTHETIC] On Colab: run the SAME pipeline for bis-His heme vs His/Met heme vs [4Fe-4S] vs Zn.")

## 3 · Cofactor docking + site pLDDT + caveated active-site MD (top candidates)
Docking (AutoDock Vina) checks the cofactor **fits and is oriented** — for heme, does the porphyrin
fit and does the Fe sit **between** the axial ligands? — *not* affinity, not incorporation, not
function. The **site pLDDT** is the trustworthy confidence (the global pLDDT can look great while the
coordinating atoms are misplaced). Short MD (OpenMM) checks the pocket doesn't drift — but **classical
metal/heme force fields are approximate**, so treat it as a weak, caveated proxy. Plot these for the
ranked survivors as orthogonal evidence.

In [ ]:
import matplotlib.pyplot as plt
# ranked.csv comes from the shared filter (fp.Design fields); vina_score/site pLDDT live in
# campaign.csv, so merge them back by design_id for the docking-vs-MD view.
try:
    ranked = pd.read_csv("results/ranked.csv")
    ranked = ranked.merge(camp[["design_id", "vina_score", "plddt_site", "fe_between_ligands"]],
                          on="design_id", how="left")
except FileNotFoundError:
    ranked = camp.copy()

top = ranked.head(min(20, len(ranked)))
fig, ax = plt.subplots(figsize=(5.4, 3.4))
sc = ax.scatter(top["vina_score"], top["md_rmsd"],
                c=top["coordination_geom_rmsd"] if "coordination_geom_rmsd" in top
                  else top["catalytic_geom_rmsd"], cmap="viridis")
ax.set_xlabel("Vina cofactor-fit score (more negative = better fit)  [SYNTHETIC]")
ax.set_ylabel("pocket MD RMSD (A) — CAVEATED metal-FF proxy  [SYNTHETIC]")
ax.set_title("Top candidates: cofactor fit vs pocket stability")
fig.colorbar(sc, label="coordination-geom RMSD (A)")
plt.tight_layout(); plt.savefig("results/docking_md.png", dpi=150); plt.show()
print("Lower-left + dark points (good fit, stable, good coordination geometry) are the best [SYNTHETIC].")
print("MD CAVEAT: classical metal/heme FF is approximate — weak proxy only; spectroscopy is the real test.")

## 4 · Honest hit-rate accounting + redox-tuning reasoning `[extension]`
Report N(pass all layers) / N(generated), and remind the reader of the field reality: even a good
preservation rate is **not** an incorporation rate, and incorporation is **not** function. Then reason
(qualitatively — **no fabricated numbers**) about **redox tuning**: how the **axial-ligand identity**
(His vs Met), **second-shell** residues, and **pocket polarity** would be expected to shift the heme
redox midpoint / O₂ affinity — the hard, valuable problem after coordination.

In [ ]:
n_total = len(camp)
try:
    ranked = pd.read_csv("results/ranked.csv")
    n_hits = int((ranked["layers_passed"] >= 3).sum())
except Exception:
    n_hits = int((camp["coordination_geom_rmsd"] <= 0.5).sum())
print("Hit-rate accounting [SYNTHETIC demo]:")
print(f"  generated             : {n_total}")
print(f"  pass all filter layers: {n_hits}  ({100*n_hits/max(n_total,1):.1f}%)")
print("\nREALITY CHECK: de novo metalloprotein hit rates are LOW, and coordination geometry does NOT")
print("guarantee cofactor incorporation, let alone the designed redox/O2 behaviour. Only spectroscopy decides.")
print("\nRedox-tuning reasoning [extension] (qualitative — NEVER fabricate a midpoint potential):")
print("  - axial-ligand identity (bis-His vs His/Met) shifts the midpoint and spin state;")
print("  - second-shell H-bonds / charges around the propionates tune the potential;")
print("  - pocket hydrophobicity/burial raises/lowers the potential; an open distal pocket -> O2 binding.")

## D3 (part 2) checklist
- [ ] Coordination-geometry preservation histogram (`results/coordination_geometry_hist.png`) + rate.
- [ ] Cofactor/scheme comparison table/figure (real schemes on Colab) `[extension]`.
- [ ] Cofactor-docking + site-pLDDT + caveated-MD figure on the ranked top set.
- [ ] Honest hit-rate accounting with the "geometry ≠ incorporation ≠ function" caveat stated.
- [ ] Redox-tuning reasoning written up (qualitative; no fabricated potentials) `[extension]`.

**Next:** `05_validation_plan.ipynb` — the spectroscopic-assay plan + cofactor titration + controls.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation plan — spectroscopic assay, cofactor titration, controls

**Standard slot:** *validation plan.* **For Project 24 this means:** turn the coordination-geometry-
filtered set into a costed **spectroscopic-assay plan** (UV-vis **Soret** band for heme / **EPR** for
Fe-S) with a **cofactor titration** and the right controls (an **apo** protein and a
**coordinating-residue→Ala** mutant), plus redox-tuning reasoning for hits (D4/D5).

This is the deliverable that states, plainly: **coordination geometry is a hypothesis; spectroscopy
tests incorporation and coordination.** **No spectra are fabricated anywhere in this project.**

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the synthesis set (< 96 designs)
Pick the top designs from the ranked filter (coordination geometry first), capped at **< 96** so they
fit a single screening plate with controls. Diversity matters — don't pick 96 near-identical designs;
spread across scaffolds (and, if you compared schemes, across cofactor schemes).

In [ ]:
import pandas as pd
try:
    ranked = pd.read_csv("results/ranked.csv")
except FileNotFoundError:
    ranked = pd.read_csv("results/campaign.csv")

# Prefer designs passing the coordination-geometry bar; cap under 96 (leave wells for controls).
geom_col = "catalytic_geom_rmsd" if "catalytic_geom_rmsd" in ranked else "coordination_geom_rmsd"
ok = ranked[ranked[geom_col] <= 0.5] if geom_col in ranked else ranked
selected = ok.head(84).copy()
selected.to_csv("results/synthesis_set.csv", index=False)
print(f"selected {len(selected)} designs for synthesis (< 96, leaving wells for controls) [SYNTHETIC ranking]")
print("Diversify across scaffolds (and schemes); record why each was chosen in your report.")

## 2 · The spectroscopic assay (the readout that confirms incorporation + coordination)
- **Express + purify:** *E. coli* BL21(DE3), 16–18 °C overnight; His-tag → IMAC → SEC polish. For
  **heme**, reconstitute the apo protein with hemin in vitro (or co-express/supplement); for
  **[4Fe-4S]**, cluster assembly is **air-sensitive** (anaerobic reconstitution).
- **Heme → UV-vis Soret band:** the intense **Soret** absorption (and Q-bands) report heme binding,
  coordination, and oxidation/spin state. Run a **heme/cofactor titration** (sub-stoichiometric →
  stoichiometric cofactor, follow the Soret to an endpoint) to confirm **1:1 binding**.
- **[4Fe-4S] / high-spin heme → EPR:** the EPR signature reports cluster/metal oxidation + spin state;
  add a redox titration if you study electron transfer.
- **Quantify the metal/cofactor:** confirm stoichiometry (pyridine-hemochrome for heme; ICP /
  colorimetric metal assays) — **incorporation is uncertain and must be measured, not assumed.**

> **Report positions/shifts qualitatively — do NOT fabricate a Soret wavelength, an extinction
> coefficient, an EPR g-value, or a redox midpoint potential.** There are no spectra in this project.

In [ ]:
controls = {
    "POSITIVE — natural reference cofactor protein": "a verified cytochrome (heme) / ferredoxin (Fe-S); "
        "confirms the assay + the expected spectroscopic signature",
    "NEGATIVE — coordinating-residue -> Ala mutant": "SAME design, an axial His (or a Cys) mutated to Ala; "
        "cleanest negative — loss of the coordinated signature pins binding to that residue",
    "NEGATIVE — apo protein (no cofactor added)": "the same protein with NO cofactor; the baseline "
        "spectrum to subtract — distinguishes specific coordination from non-specific cofactor sticking",
    "BLANK — buffer + cofactor only": "free-cofactor spectrum (free heme/hemin has its own Soret) to "
        "subtract; separates bound from unbound cofactor",
}
print("MANDATORY controls (every experiment):")
for k, v in controls.items():
    print(f"  - {k}\n      {v}")

## 3 · A costed spectroscopic screen (template — fill real prices)
Cost the gene synthesis, expression, cofactor reconstitution, and spectroscopy at your institution's
rates; the cell prints a template to fill in your report. The <96 designs + controls fit a plate for
expression; the UV-vis/EPR readout is then per-sample.

In [ ]:
plan = [
    ("Gene synthesis (codon-optimised, screened provider)", "< 96 designs", "fill price/construct"),
    ("Cloning + transformation", "1 plate", "fill"),
    ("Expression + lysis", "1 plate", "fill"),
    ("IMAC + SEC purification", "per design", "fill"),
    ("Cofactor reconstitution (hemin / anaerobic Fe-S assembly)", "per design", "fill"),
    ("UV-vis spectrophotometer time (Soret titration)", "per sample", "fill"),
    ("EPR time (Fe-S / high-spin heme)", "per sample", "fill (if applicable)"),
    ("Metal/cofactor quantification (pyridine-hemochrome / ICP)", "per hit", "fill"),
]
print("Costed reagent/step list (fill institutional prices) [TEMPLATE]:")
for step, scale, cost in plan:
    print(f"  - {step:56s} {scale:14s} {cost}")
print("\nTimeline (typical): synthesis 2-3 wk -> clone/express 1-2 wk -> purify+reconstitute 1-2 wk")
print("  -> UV-vis/EPR titration 1-2 wk. Fe-S adds anaerobic-handling overhead.")
print("Synthesis MUST go through an IGSC-member, biosecurity-screening provider (low dual-use here, "
      "but it is policy). Wet-lab needs institutional biosafety/ethics sign-off.")

## 4 · Redox-tuning / function engineering `[extension]`/`[stretch]`
Coordination is the start; **tuning the function** is the hard, valuable problem after it. For a
coordinating design, plan to test (with the **same spectroscopic readout**) how:
- **axial-ligand identity** (His vs Met; bis-His vs proximal-His-open) shifts the redox midpoint / spin
  state / O₂ affinity;
- **second-shell** residues (H-bonds to the propionates, charges near the Fe) tune the midpoint;
- **pocket polarity / burial** raises or lowers the potential.

Plan a small mutant series around the coordinating site and read the shift by UV-vis/EPR (and, for
electron transfer, redox potentiometry). **Report shifts qualitatively / as measured — never fabricate
a midpoint potential.**

In [ ]:
print("Redox-tuning loop (stretch): pick a coordinating design -> mutate axial ligand / second shell /")
print("pocket polarity -> express + reconstitute -> read the Soret/EPR shift (and redox potentiometry")
print("for electron transfer) -> iterate. Cite designed-heme-protein + maquette redox-tuning literature.")
print("Reminder for the thesis: report incorporation fraction + the spectroscopic trajectory honestly,")
print("not just the best-looking spectrum; and NEVER fabricate a wavelength, g-value, or potential.")

## D4 / D5 checklist
- [ ] `results/synthesis_set.csv`: < 96 diverse, coordination-geometry-passing designs.
- [ ] Spectroscopic-assay plan: UV-vis **Soret** (heme) / **EPR** (Fe-S) + a **cofactor titration**,
      with **all** controls (coordinating-residue→Ala mutant, apo protein, blank), costed + timed.
- [ ] Metal/cofactor quantification planned (incorporation measured, not assumed).
- [ ] Redox-tuning / function-engineering plan for hits `[extension]`/`[stretch]`.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; "coordination geometry ≠ incorporation ≠
      function" stated plainly; **no fabricated spectra**.

You're done — this project specialises the Project-18 enzyme-family template
(cofactor-spec → scaffold → cofactor-aware sequence → coordination-geometry) to a **cofactor-
coordination site** validated by **spectroscopy**.